# NB01. Data Preparation

**목적**: 분석용 데이터셋을 구성하고 단일 train/test split을 생성한다. NB01-08은 분석 흐름을 잡는 단계이며, k-fold CV는 최종 실험에서 별도 수행한다.

**Dependencies**: None (entry point)

**Outputs** (`outputs/NB01/`):
- `datasets.pkl`: {name: {X_train, X_test, y_train, y_test, feature_names, metadata}}
- `data_summary.csv`: 데이터셋 요약
- `correlations.pkl`: 상관행렬

**Design Decisions**:
- 단일 train/test split (80/20, stratified, RS=317)
- 임퓨테이션: train median → train/test 모두에 적용 (누출 방지)
- HC: 결측 40%+ 제거 → sparse binary (소수<1%, IV<0.02) 제거
- GMSC: age 도메인 클리핑 (18-100)

**Dataset Sources**:
- GMSC: Kaggle "Give Me Some Credit" (2011)
- HC: Kaggle "Home Credit Default Risk" (2018)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 317
TEST_SIZE = 0.2
DATA_DIR = Path('../.data')
OUT_DIR = Path('../outputs/NB01')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Sparse Binary Filter Utility

In [2]:
def compute_iv_binary(x, y, eps=1e-8):
    """Information Value for a binary feature."""
    x, y = np.asarray(x), np.asarray(y)
    vals = np.unique(x[~np.isnan(x)])
    if len(vals) != 2:
        return 0.0
    iv = 0.0
    n_good, n_bad = (y == 0).sum(), (y == 1).sum()
    for v in vals:
        mask = x == v
        g, b = ((y == 0) & mask).sum(), ((y == 1) & mask).sum()
        dr_g = (g + eps) / (n_good + eps)
        dr_b = (b + eps) / (n_bad + eps)
        iv += (dr_g - dr_b) * np.log(dr_g / dr_b)
    return float(iv)

def filter_sparse_binaries(X, y, min_rate=0.01, min_iv=0.02):
    """Drop binary features with <min_rate minority AND IV < min_iv."""
    drop_cols, drop_info = [], []
    for col in X.columns:
        vc = X[col].dropna().value_counts(normalize=True)
        if len(vc) == 2 and vc.min() < min_rate:
            iv = compute_iv_binary(X[col].values, y.values)
            action = 'DROPPED' if iv < min_iv else 'KEPT (IV >= 0.02)'
            if iv < min_iv:
                drop_cols.append(col)
            drop_info.append({'column': col, 'minority_rate': f'{vc.min():.3%}',
                              'IV': round(iv, 4), 'action': action})
    return X.drop(columns=drop_cols), drop_info

## 2. GMSC

In [3]:
df = pd.read_csv(DATA_DIR / 'give_me_some_credit/cs-training.csv', index_col=0)
y_gmsc = df['SeriousDlqin2yrs']
X_gmsc = df.drop('SeriousDlqin2yrs', axis=1).select_dtypes(include=[np.number])

# Domain clip
n_clip = ((X_gmsc['age'] < 18) | (X_gmsc['age'] > 100)).sum()
X_gmsc['age'] = X_gmsc['age'].clip(18, 100)

print(f'GMSC: {X_gmsc.shape[0]:,} x {X_gmsc.shape[1]}, default {y_gmsc.mean():.1%}, age clipped: {n_clip}')

GMSC: 150,000 x 10, default 6.7%, age clipped: 14


## 3. HC

In [4]:
df = pd.read_csv(DATA_DIR / 'home_credit/application_train.csv')
y_hc = df['TARGET']
X_hc = df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore').select_dtypes(include=[np.number])
n_orig = X_hc.shape[1]

# Step 1: 결측 40%+ 제거
X_hc = X_hc[X_hc.columns[X_hc.isnull().mean() < 0.4]]
n_after_miss = X_hc.shape[1]

# Step 2: sparse binary (소수<1%, IV<0.02) 제거
X_hc, sparse_info = filter_sparse_binaries(X_hc, y_hc)
n_final = X_hc.shape[1]

print(f'HC: {n_orig} → {n_after_miss} (miss>40%) → {n_final} (sparse binary)')
print(f'\nDropped sparse binaries ({n_after_miss - n_final}):')
for info in sparse_info:
    print(f'  {info["column"]:>30s}  min={info["minority_rate"]}  IV={info["IV"]:.4f}  {info["action"]}')

HC: 104 → 59 (miss>40%) → 41 (sparse binary)

Dropped sparse binaries (18):
                      FLAG_MOBIL  min=0.000%  IV=0.0001  DROPPED
                FLAG_CONT_MOBILE  min=0.187%  IV=0.0000  DROPPED
                 FLAG_DOCUMENT_2  min=0.004%  IV=0.0002  DROPPED
                 FLAG_DOCUMENT_4  min=0.008%  IV=0.0017  DROPPED
                 FLAG_DOCUMENT_7  min=0.019%  IV=0.0000  DROPPED
                 FLAG_DOCUMENT_9  min=0.390%  IV=0.0003  DROPPED
                FLAG_DOCUMENT_10  min=0.002%  IV=0.0004  DROPPED
                FLAG_DOCUMENT_11  min=0.391%  IV=0.0003  DROPPED
                FLAG_DOCUMENT_12  min=0.001%  IV=0.0001  DROPPED
                FLAG_DOCUMENT_13  min=0.353%  IV=0.0028  DROPPED
                FLAG_DOCUMENT_14  min=0.294%  IV=0.0018  DROPPED
                FLAG_DOCUMENT_15  min=0.121%  IV=0.0009  DROPPED
                FLAG_DOCUMENT_16  min=0.993%  IV=0.0023  DROPPED
                FLAG_DOCUMENT_17  min=0.027%  IV=0.0003  DROPPED
              

## 4. Train/Test Split & Imputation

단일 80/20 split. Imputation은 train median을 계산하여 train/test 모두에 적용한다.

In [5]:
datasets = {}

for name, X, y in [('GMSC', X_gmsc, y_gmsc), ('HC', X_hc, y_hc)]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    
    # Impute: train median → both
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_te = X_te.fillna(medians)
    
    datasets[name] = {
        'X_train': X_tr, 'X_test': X_te,
        'y_train': y_tr, 'y_test': y_te,
        'feature_names': list(X_tr.columns),
        'train_medians': medians,
        'metadata': {
            'source': 'Kaggle GMSC 2011' if name == 'GMSC' else 'Kaggle HC 2018',
            'random_state': RANDOM_STATE,
            'test_size': TEST_SIZE,
            'generated': datetime.now().isoformat(),
        },
    }
    
    print(f'{name}: train={len(X_tr):,} ({y_tr.mean():.2%}), '
          f'test={len(X_te):,} ({y_te.mean():.2%}), '
          f'features={X_tr.shape[1]}, missing={X_tr.isnull().sum().sum() + X_te.isnull().sum().sum()}')

GMSC: train=120,000 (6.68%), test=30,000 (6.68%), features=10, missing=0


HC: train=246,008 (8.07%), test=61,503 (8.07%), features=41, missing=0


## 5. Correlation Structure

Ridge 실패 원인(다중공선성)의 사전 파악. 제거는 하지 않되 문서화한다.

In [6]:
correlations = {}
for name in datasets:
    corr = datasets[name]['X_train'].corr()
    correlations[name] = corr
    upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    
    high = [(i, j, corr.loc[i, j]) for i, j in combinations(corr.columns, 2) 
            if abs(corr.loc[i, j]) > 0.9]
    high.sort(key=lambda x: abs(x[2]), reverse=True)
    
    print(f'{name}: Mean |corr|={upper.stack().mean():.3f}, pairs |r|>0.9: {len(high)}')
    for f1, f2, r in high[:5]:
        print(f'  {f1} <-> {f2}: r={r:.4f}')

GMSC: Mean |corr|=0.116, pairs |r|>0.9: 3
  NumberOfTimes90DaysLate <-> NumberOfTime60-89DaysPastDueNotWorse: r=0.9925
  NumberOfTime30-59DaysPastDueNotWorse <-> NumberOfTime60-89DaysPastDueNotWorse: r=0.9866
  NumberOfTime30-59DaysPastDueNotWorse <-> NumberOfTimes90DaysLate: r=0.9830


HC: Mean |corr|=0.061, pairs |r|>0.9: 4
  DAYS_EMPLOYED <-> FLAG_EMP_PHONE: r=-0.9998
  OBS_30_CNT_SOCIAL_CIRCLE <-> OBS_60_CNT_SOCIAL_CIRCLE: r=0.9985
  AMT_CREDIT <-> AMT_GOODS_PRICE: r=0.9868
  REGION_RATING_CLIENT <-> REGION_RATING_CLIENT_W_CITY: r=0.9516


## 6. Save

In [7]:
with open(OUT_DIR / 'datasets.pkl', 'wb') as f:
    pickle.dump(datasets, f)
with open(OUT_DIR / 'correlations.pkl', 'wb') as f:
    pickle.dump(correlations, f)

# Summary table
rows = []
for name in datasets:
    d = datasets[name]
    upper = correlations[name].abs().where(
        np.triu(np.ones(correlations[name].shape), k=1).astype(bool))
    rows.append({
        'Dataset': name,
        'Train': len(d['X_train']), 'Test': len(d['X_test']),
        'Features': d['X_train'].shape[1],
        'Default': f'{d["y_train"].mean():.1%}',
        'Mean|r|': f'{upper.stack().mean():.3f}',
        '|r|>0.7': int((upper.stack() > 0.7).sum()),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / 'data_summary.csv', index=False)
print(summary.to_string(index=False))

print(f'\nSaved to {OUT_DIR}/')
for p in sorted(OUT_DIR.iterdir()):
    s = p.stat().st_size
    print(f'  {p.name} ({s/1024/1024:.1f} MB)' if s > 1e6 else f'  {p.name} ({s/1024:.0f} KB)')

Dataset  Train  Test  Features Default Mean|r|  |r|>0.7
   GMSC 120000 30000        10    6.7%   0.116        3
     HC 246008 61503        41    8.1%   0.061       10

Saved to ..\outputs\NB01/
  correlations.pkl (16 KB)
  cv_splits.pkl (19.3 MB)
  data_summary.csv (0 KB)
  datasets.pkl (121.6 MB)
